
# Assignment 1: Boolean Model, TF-IDF, and Data Retrieval vs. Information Retrieval Conceptual Questions

**Student names**: Kirsten Sunde Thaulow and Nora Marie Thon Børaas <br>
**Group number**: 3 <br>
**Date**: 13.09.2025

## Important notes
Please carefully read the following notes and consider them for the assignment delivery. Submissions that do not fulfill these requirements will not be assessed and should be submitted again.
1. You may work in groups of maximum 2 students.
2. The assignment must be delivered in ipynb format.
3. The assignment must be typed. Handwritten assignments are not accepted.

**Due date**: 14.09.2025 23:59

In this assignment, you will:
- Implement a Boolean retrieval model
- Compute TF-IDF vectors for documents
- Run retrieval on queries
- Answer conceptual questions 

---
## Dataset

You will use the **Cranfield** dataset, provided in this file:

- `cran.all.1400`: The document collection (1400 documents)

**The code to parse the file is ready — just update the cran file path to match your own file location. Use the docs variable in your code for the parsed file**

### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.


In [1]:

# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"cran.all.1400"  # <-- change this!

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)



FileNotFoundError: [Errno 2] No such file or directory: 'cran.all.1400'

## 1.1 – Boolean Retrieval Model

### 1.1.1 Tokenize documents

Implement tokenization using the given list of stopwords. Create a list of normalized terms per document (e.g., lowercase, remove punctuation/digits; drop stopwords). Store the token lists to use in later steps.

In [ ]:
# TODO: Implement tokenization using the given list of stopwords, create list of terms per document
import re

from typing import Dict
docs: Dict[int, Dict[str, str]] = parse_cranfield(CRAN_PATH)
doc_terms: Dict[int, list[str]] = {}

STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

# Your code here


WORD_RE = re.compile(r"[A-Za-zÆØÅæøå]+(?:'[A-Za-zÆØÅæøå]+)?")

doc_terms = {}

for current_id, doc in docs.items():
    text = (doc["title"] + " " + doc["abstract"]).lower()

    tokens = WORD_RE.findall(text)

    updated_tokens = []
    for t in tokens:
        if t not in STOPWORDS:
            updated_tokens.append(t)
    
    doc_terms[current_id] = updated_tokens

print(doc_terms[1][:20])

Parsed 1400 documents.
['experimental', 'investigation', 'aerodynamics', 'wing', 'slipstream', 'experimental', 'investigation', 'aerodynamics', 'wing', 'slipstream', 'experimental', 'study', 'wing', 'propeller', 'slipstream', 'made', 'order', 'determine', 'spanwise', 'distribution']


### Build vocabulary

Create a set (or list) of unique terms from all tokenized documents. Report the number of unique terms.


In [ ]:
# TODO: Create a set or list of unique terms

# Report: 
# - Number of unique terms

# Your code here


unique_terms = set()
for current_id, term in doc_terms.items():
    for t in term:
        unique_terms.add(t)
print(f"Number of unique terms: {len(unique_terms)}")  

Number of unique terms: 7036


### Build inverted index

For each term, store the list (or set) of document IDs where the term appears.


In [ ]:

# TODO: For each term, store list of document IDs where the term appears
# Your code here

term_docs = {}

for current_id, terms in doc_terms.items():
    for term in terms:
        if term not in term_docs:
            term_docs[term] = []
        term_docs[term].append(current_id)

print(f"Number of terms with document IDs: {len(term_docs)}")
print(f"Example term with document IDs: {list(term_docs.items())[0]}")

Number of terms with document IDs: 7036
Example term with document IDs: ('experimental', [1, 1, 1, 11, 12, 17, 19, 25, 29, 30, 35, 41, 42, 47, 52, 52, 53, 58, 69, 70, 74, 74, 78, 84, 84, 84, 99, 99, 101, 103, 112, 115, 121, 123, 123, 137, 140, 142, 154, 156, 168, 170, 171, 173, 173, 176, 179, 179, 183, 184, 186, 186, 187, 188, 189, 189, 191, 195, 195, 195, 197, 197, 202, 203, 206, 206, 207, 207, 212, 216, 220, 222, 225, 227, 230, 234, 234, 234, 234, 245, 251, 256, 256, 256, 257, 262, 271, 271, 271, 273, 277, 282, 283, 286, 294, 295, 304, 307, 329, 329, 330, 334, 334, 338, 339, 339, 344, 344, 344, 345, 346, 346, 347, 354, 360, 369, 370, 372, 372, 372, 377, 397, 409, 411, 411, 413, 413, 418, 420, 420, 421, 423, 423, 427, 435, 439, 441, 442, 442, 453, 455, 455, 462, 464, 467, 484, 484, 484, 494, 494, 496, 497, 497, 498, 501, 503, 504, 505, 511, 518, 520, 520, 522, 522, 522, 536, 540, 544, 544, 544, 549, 549, 552, 552, 553, 558, 558, 563, 567, 569, 569, 572, 572, 572, 572, 576, 588, 595, 6

### Retrieve documents for a Boolean query (AND/OR)

Create a function to retrieve documents for a Boolean query (AND/OR) with query terms.  


In [ ]:
def boolean_retrieve(query: str):
    tokens = query.split()
    if not tokens:
        return []

    def postings(term):
        return set(term_docs.get(term, []))

    result = postings(tokens[0])

    i = 1
    while i + 1 < len(tokens):
        op = tokens[i]
        term = tokens[i + 1]
        p = postings(term)

        if op == "AND":
            result &= p
        elif op == "OR":
            result |= p

        i += 2

    return sorted(result)


In [ ]:
# Do not change this code
boolean_queries = [
  "gas AND pressure",
  "structural AND aeroelastic AND flight AND high AND speed OR aircraft",
  "heat AND conduction AND composite AND slabs",
  "boundary AND layer AND control",
  "compressible AND flow AND nozzle",
  "combustion AND chamber AND injection",
  "laminar AND turbulent AND transition",
  "fatigue AND crack AND growth",
  "wing AND tip AND vortices",
  "propulsion AND efficiency"
]

In [ ]:
# Run Boolean queries in batch, using the function you created
def run_batch_boolean(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = boolean_retrieve(q)
        results[f"Q{i}"] = res
    return results

boolean_results = run_batch_boolean(boolean_queries)
for qid, res in boolean_results.items():
    print(qid, "=>", res[:5])


Q1 => [27, 49, 85, 101, 110]
Q2 => [12, 14, 29, 47, 51]
Q3 => [5, 399]
Q4 => [1, 61, 244, 265, 342]
Q5 => [118, 131]
Q6 => []
Q7 => [7, 9, 80, 89, 96]
Q8 => []
Q9 => [675]
Q10 => [968]


## Part 1.2 – TF-IDF Indexing


$tf_{i,j} = \text{Raw Frequency}$

$idf_t = \log\left(\frac{N}{df_t}\right)$

### Build document–term matrix (TF and IDF weights)

Compute tf and idf using the formulas above and store the weights in a document–term matrix (rows = documents, columns = terms).



In [ ]:
# TODO: Calculate the weights for the documents and the terms using tf and idf weighting. Put these values into a document–term matrix (rows = documents, columns = terms).
# Your code here
import math
from collections import Counter


N = len(docs)
vocabulary = sorted(unique_terms)
col_index = {t: i for i, t in enumerate(vocabulary)}

#Making sure all the document_ids are unique for each term
df = {t: len(set(term_docs[t])) for t in vocabulary}


#IDF_T = log(N / df_t)
idf = {t: math.log(N / df[t]) if df[t] > 0 else 0.0 for t in vocabulary}

#Building the Matrix
doc_ids = sorted(docs.keys())                 
M = [[0.0] * len(vocabulary) for _ in range(N)]   
row_index = {doc_id: r for r, doc_id in enumerate(doc_ids)}

for doc_id, tokens in doc_terms.items():
    r = row_index[doc_id]
    tf = Counter(tokens)
    for t, count in tf.items():
        if count > 0 and t in col_index:
            w_tf  = 1 + math.log(count)          
            w_idf = idf[t]                       
            M[r][col_index[t]] = w_tf * w_idf

# (optional quick checks)
print(len(M), len(M[0]))         # expect: N x |vocabulary|
print(sum(x > 0 for row in M for x in row))


1400 7036
90492


### Build TF–IDF document vectors

From the matrix, build a TF–IDF vector for each document (consider normalization if needed for cosine similarity).


In [ ]:

# TODO: Build TF–IDF document vectors from the document–term matrix
# Your code here

doc_vectors = {}

for doc_id in doc_ids:
    doc_vectors[doc_id] = M[row_index[doc_id]]
    norm = math.sqrt(sum(x*x for x in doc_vectors[doc_id]))
    if norm > 0:
        doc_vectors[doc_id] = [x / norm for x in doc_vectors[doc_id]]


vec = doc_vectors[1]
print([x for x in vec if abs(x) > 1e-12][:20])

[0.19963001560098617, 0.1095629370991853, 0.07770705347836017, 0.07323722716447328, 0.08770621197346117, 0.03233621071636222, 0.1581023526891131, 0.10167781795171284, 0.09967726788739543, 0.08901454554121888, 0.3986486890537571, 0.07594915315532426, 0.1570470622149375, 0.05065856080881189, 0.11200517642529767, 0.047615363501240356, 0.04225743661546534, 0.11143432670553234, 0.19963001560098617, 0.11343487676984974]


### Implement cosine similarity

Implement a function to compute cosine similarity scores between a (tokenized) query and all documents.


In [ ]:

# TODO: Create a function for calculating the similarity score of all the documents by their relevance to query terms

def tfidf_retrieve(query: str):
    # Your code here

    #Tokenizing? Unsure. Creates zero-vector
    tokens = [t for t in WORD_RE.findall(query.lower()) if t not in STOPWORDS]
    tf_q = Counter(tokens)
    q_vec = [0.0] * len(vocabulary)

    #Add weights to terms that appear in the query in the q_vec
    for t, c in tf_q.items():
        if t in col_index and c > 0:
            q_vec[col_index[t]] = (1 + math.log(c)) * idf[t]
    
    #normalizing the vector
    norm = math.sqrt(sum(x*x for x in q_vec))
    if norm > 0:
        q_vec = [x / norm for x in q_vec]
    
    relevant_vectors = {}
    for doc_id, doc in doc_vectors.items():
        relevant_vectors[doc_id] = sum(a*b for a, b in zip(doc_vectors[doc_id], q_vec))
    
    sorted_docs = dict(sorted(relevant_vectors.items(), key=lambda kv: kv[1], reverse=True))
    sorted_list = list(sorted_docs.keys())

        
    return sorted_list


In [ ]:
# Do not change this code
tfidf_queries = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency"
]

In [ ]:
# Run TF-IDF queries in batch (print top-5 results for each), using the function you created
def run_batch_tfidf(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = tfidf_retrieve(q)
        results[f"Q{i}"] = res
    return results

tfidf_results = run_batch_tfidf(tfidf_queries)

for qid, res in tfidf_results.items():
    print(qid, "=>", res[:5])


Q1 => [665, 1003, 1312, 169, 337]
Q2 => [12, 875, 746, 51, 884]
Q3 => [399, 485, 5, 144, 181]
Q4 => [748, 451, 368, 265, 1349]
Q5 => [389, 118, 965, 1353, 1187]
Q6 => [1143, 1254, 1241, 635, 1269]
Q7 => [418, 337, 1264, 959, 96]
Q8 => [768, 1196, 726, 884, 883]
Q9 => [675, 1284, 288, 433, 434]
Q10 => [968, 1328, 1380, 1092, 592]



## Part 1.3 – Conceptual Questions

Answer the following questions:

**1. What is the difference between data retrieval and information retrieval?**  
*Data retrieval is all about retrieving data from a structured set of data saved in databases. The user will specify certain criterias. The answer will then be exact data sets that matches the query. Information retrieval is about retrieving data in an unstructured or semi structured nature. Like documents and web sites. The user specifies human like language and key words. The answer is then a ranged list of the most relevant documents. Relevance is above exact answers.*

**For the following scenarios, which approach would be suitable data retrieval or information retrieval? Explain your reasoning.** <br>
1.a A clerk in pharmacy uses the following query: Medicine_name = Ibuprofen_400mg <br>
*data retrieval. In such a scenario it is only the one type of medicine the clerk is looking for. It is for example not relevant for him to find other medicines that are 400 mg.*

1.b A clerk in pharmacy uses the following query: An anti-biotic medicine <br>
*Here information retrieval is a better fit. Because no medicines are has the exact same name. If you would search for this in a data base each medicine would need to have a category that matches exactly this query which is not optimal.*

1.c Searching for the schedule of a flight using the following query: Flight_ID = ZEFV2 <br>
*Here we are only interested in one specific flight_ID. It is not relevant for the person to find other flight_IDs that are similar. This does not indicate that the flight_ID actually has anything to do with our wanted flight_ID. Data retrieval.*

1.d Searching an E-commerce website using the following query to find an specific shoe: Brooks Ghost 15 <br>
*Even though the query is a pretty specific one, it is, in this scenrio still relevant to fins other shoe models that match. If the E-commerce website does not have Brooks Ghost 15, maybe the buyer would consider buying Brooks Ghost 14. Therefore information retrieval.*

1.e Searching the same E-commerce website using the following query: Nice running shoes <br>
*This is a broad search using subjective descriptions as running and nice. It is therefore better using information retrieval. It is not realistic that any shoe model would have the exact name of "nice running shoes"*
